# 🏦 ¡Enséñame la Pasta! — Predicción de Riesgo Crediticio

**Objetivo:** Predecir `SeriousDlqin2yrs` (1 = impago grave en 2 años, 0 = no impago)

**Métrica:** La competición típicamente usa **AUC-ROC** (área bajo la curva ROC).

---
## 📋 Resumen del dataset
- **Train:** 105.000 filas × 12 columnas (incluye target)
- **Test:** 45.000 filas × 11 columnas (sin target — esto es lo que hay que predecir)
- **Clase desbalanceada:** solo ~6.7% de casos positivos (impago)
- **Nulls:** `MonthlyIncome` (~20k nulls), `NumberOfDependents` (~2.7k nulls)
- **Outliers notables:** valores 98 en columnas de días de retraso (error de datos), RevolvingUtilization y DebtRatio con valores extremos

## 1. 📦 Imports y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression


# Carga 
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')

print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head()

Train: (105000, 12)  |  Test: (45000, 11)


,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,9580,0.668999,58,2,0.449504,3425.0,9,1,1,1,1.0,0
1,39755,0.015922,71,0,6.000000,NaN,5,0,0,0,0.0,0
2,118799,0.183062,52,1,0.035593,5000.0,9,0,0,0,0.0,0
3,16489,0.162301,77,0,0.227886,2000.0,8,0,0,0,0.0,0
4,149857,0.404199,30,0,0.026010,5843.0,4,0,0,0,0.0,0


In [2]:
from ydata_profiling import ProfileReport
profile = ProfileReport(train, title='Prestamos Profiling Report')
profile.to_file('prestamos_report.html')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 47.03it/s]


## 2. 🔍 EDA — Análisis Exploratorio

In [ ]:
# --- 2.1 Variable objetivo: muy desbalanceada ---
print("📊 Distribución del target:")
print(train['SeriousDlqin2yrs'].value_counts())
print(train['SeriousDlqin2yrs'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(5,3))
train['SeriousDlqin2yrs'].value_counts().plot(kind='bar', ax=ax, color=['steelblue','coral'])
ax.set_title('Distribución del Target (0=Ok, 1=Impago)')
ax.set_xticklabels(['Sin impago', 'Impago grave'], rotation=0)
plt.tight_layout()
plt.show()
print("\n⚠️  Dataset desbalanceado (~6.7% positivos). Tenlo en cuenta al evaluar modelos.")

In [ ]:
# --- 2.2 Valores nulos ---
nulls = train.isnull().sum().sort_values(ascending=False)
nulls_pct = (nulls / len(train) * 100).round(2)
print("Valores nulos en TRAIN:")
print(pd.DataFrame({'Count': nulls, '%': nulls_pct})[nulls > 0])

In [ ]:
# --- 2.3 Distribución de features numéricas ---
features = [c for c in train.columns if c not in ['ID', 'SeriousDlqin2yrs']]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(features):
    train[col].clip(upper=train[col].quantile(0.99)).hist(ax=axes[i], bins=30, color='steelblue', alpha=0.7)
    axes[i].set_title(col, fontsize=8)
    axes[i].set_xlabel('')
# Ocultar subplots sobrantes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribución de variables (cap al percentil 99)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.4 Correlación con el target ---
corrs = train[features].corrwith(train['SeriousDlqin2yrs']).sort_values()
fig, ax = plt.subplots(figsize=(7,4))
corrs.plot(kind='barh', ax=ax, color=['coral' if v > 0 else 'steelblue' for v in corrs])
ax.set_title('Correlación con SeriousDlqin2yrs')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()
print("Variables más correlacionadas con impago (positivo = más riesgo):")
print(corrs.sort_values(ascending=False).head(5))

## 3. 🧹 Limpieza y Preprocesamiento

In [ ]:
def preprocess(df, is_train=True):
    df = df.copy()

    # --- 3.1 Tratar outliers en columnas de retrasos ---
    # El valor 98 es un código de error conocido en este dataset
    late_cols = [
        'NumberOfTime30-59DaysPastDueNotWorse',
        'NumberOfTimes90DaysLate',
        'NumberOfTime60-89DaysPastDueNotWorse'
    ]
    for col in late_cols:
        df[col] = df[col].replace(98, np.nan)  # tratar como nulo
        df[col] = df[col].fillna(df[col].median())

    # --- 3.2 Imputar nulos ---
    # MonthlyIncome: mediana (distribución muy sesgada)
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
    # NumberOfDependents: mediana
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

    # --- 3.3 Winsorize / cap outliers extremos ---
    # RevolvingUtilization puede ser > 1 (uso > límite de crédito), pero 29110 es absurdo
    df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.5)
    df['DebtRatio'] = df['DebtRatio'].clip(upper=df['DebtRatio'].quantile(0.99))
    df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=df['MonthlyIncome'].quantile(0.99))

    # --- 3.4 Feature Engineering sencillo ---
    # Total de veces con retraso (suma de todas las columnas de retraso)
    df['TotalLate'] = df[late_cols].sum(axis=1)
    # Ratio deuda/ingresos
    df['DebtToIncome'] = df['DebtRatio'] / (df['MonthlyIncome'] + 1)
    # Flag de ingreso bajo
    df['LowIncome'] = (df['MonthlyIncome'] < 3000).astype(int)

    return df

train_clean = preprocess(train)
test_clean  = preprocess(test)

print("✅ Preprocesamiento completo")
print(f"Train nulls restantes: {train_clean.isnull().sum().sum()}")
print(f"Test nulls restantes:  {test_clean.isnull().sum().sum()}")

## 4. ✂️ Split Train / Validation

In [ ]:
FEATURE_COLS = [c for c in train_clean.columns if c not in ['ID', 'SeriousDlqin2yrs']]
TARGET = 'SeriousDlqin2yrs'

X = train_clean[FEATURE_COLS]
y = train_clean[TARGET]
X_kaggle = test_clean[FEATURE_COLS]

# Stratified split para mantener proporción de clases
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")
print(f"Positivos en train: {y_train.mean():.3f}  |  en val: {y_val.mean():.3f}")
print(f"\nFeatures usadas ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)

## 5. 🤖 Modelos

Probaremos varios modelos de menor a mayor complejidad:
1. **Logistic Regression** — baseline simple
2. **Random Forest** — ensemble de árboles
3. **LightGBM** — gradient boosting, suele ser el mejor en tabular

In [ ]:
# Helper: evaluar modelo
def evaluate(name, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_v)[:, 1]
    auc = roc_auc_score(y_v, proba)
    print(f"  [{name}] AUC-ROC en validación: {auc:.4f}")
    return auc, proba

results = {}

In [ ]:
# --- 5.1 Logistic Regression (baseline) ---
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',  # ← importante por el desbalanceo
        max_iter=1000,
        random_state=42
    ))
])

auc_lr, proba_lr = evaluate("Logistic Regression", lr_pipe, X_train, y_train, X_val, y_val)
results['Logistic Regression'] = auc_lr

In [ ]:
# --- 5.2 Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

auc_rf, proba_rf = evaluate("Random Forest", rf, X_train, y_train, X_val, y_val)
results['Random Forest'] = auc_rf

In [ ]:
# --- 5.3 LightGBM (recomendado) ---
if HAS_LGB:
    # Calcular scale_pos_weight para desbalanceo
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = neg / pos
    print(f"scale_pos_weight = {spw:.1f}")

    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        scale_pos_weight=spw,  # manejo del desbalanceo
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

    auc_lgb, proba_lgb = evaluate("LightGBM", lgb_model, X_train, y_train, X_val, y_val)
    results['LightGBM'] = auc_lgb
else:
    print("Instala LightGBM para mejor resultado: pip install lightgbm")

In [ ]:
# --- 5.4 Comparativa de modelos ---
print("\n📊 RESUMEN DE MODELOS")
print("-" * 40)
for name, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(auc * 100 - 50)  # barra visual relativa desde 0.5
    print(f"  {name:25s} AUC: {auc:.4f}  {bar}")

best_model_name = max(results, key=results.get)
print(f"\n🏆 Mejor modelo: {best_model_name} ({results[best_model_name]:.4f})")

In [ ]:
# --- 5.5 Curvas ROC ---
fig, ax = plt.subplots(figsize=(7,5))

model_probas = {
    'Logistic Regression': proba_lr,
    'Random Forest': proba_rf,
}
if HAS_LGB:
    model_probas['LightGBM'] = proba_lgb

for name, proba in model_probas.items():
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random (0.5000)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.6 Feature Importance (Random Forest / LightGBM) ---
model_for_importance = lgb_model if HAS_LGB else rf
name_for_importance = 'LightGBM' if HAS_LGB else 'Random Forest'

fi = pd.Series(
    model_for_importance.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
fi.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Feature Importance — {name_for_importance}')
plt.tight_layout()
plt.show()

## 6. 📤 Generar Submission para Kaggle

In [ ]:
# Reentrenar el mejor modelo con TODOS los datos de train (no solo el 80%)
best_model_map = {
    'Logistic Regression': lr_pipe,
    'Random Forest': rf,
}
if HAS_LGB:
    best_model_map['LightGBM'] = lgb_model

best_model = best_model_map[best_model_name]

print(f"Reentrenando {best_model_name} con el dataset completo...")
best_model.fit(X, y)

# Predecir probabilidades en test de Kaggle
test_proba = best_model.predict_proba(X_kaggle)[:, 1]

# Crear submission
submission = pd.DataFrame({
    'ID': test_clean['ID'],
    'SeriousDlqin2yrs': test_proba
})

submission.to_csv('submission.csv', index=False)

print("✅ submission.csv guardado!")
print(f"Filas: {len(submission)}")
print(f"Distribución de probabilidades:")
print(submission['SeriousDlqin2yrs'].describe())
submission.head(10)

## 7. 💡 Ideas para Mejorar (si tienes tiempo)

### Feature Engineering
- **Ratio de utilización de crédito** ya está en los datos (`RevolvingUtilization`), pero puedes crear combinaciones
- **Interacciones**: `Age * TotalLate`, etc.
- **Bins de edad**: joven (<30), adulto (30-50), mayor (>50)

### Manejo del desbalanceo
- **SMOTE** (oversample de la clase minoritaria): `pip install imbalanced-learn`
- Ajustar `class_weight` o `scale_pos_weight`

### Hyperparameter tuning
```python
# Ejemplo rápido con Optuna (pip install optuna)
import optuna
# o simplemente probar manualmente distintos learning_rate, n_estimators, max_depth
```

### Cross-Validation más robusta
```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lgb_model, X, y, cv=cv, scoring='roc_auc')
print(f"CV AUC: {scores.mean():.4f} ± {scores.std():.4f}")
```

### Ensemble / Stacking
- Promediar las probabilidades de LightGBM + Random Forest puede subir ligeramente el AUC

---

> **Recuerda:** Para Kaggle, el submission debe tener exactamente las columnas `ID` y `SeriousDlqin2yrs` con las **probabilidades** (no 0/1), ya que la métrica es AUC-ROC.